# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities via their unique `@id` as prescribed by the Croissant schema.

### Dataset Source
The dataset and its Croissant metadata can be accessed at:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

#### Dataset Overview
- **Title**: Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review
- **Description**: Structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients with non-classical 21-hydroxylase deficiency-associated infertility.
- **License**: https://opendatacommons.org/licenses/by/1-0/
- **Identifier**: 10.71728/senscience.cbsv-djdb

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Since Croissant schema organizes data in record sets, we list their `@id`s and preview the contained records and fields. All references are via `@id` fields.

_Note: If you are unsure about the available recordSet IDs or field IDs, you can access them from the metadata as_:

- `dataset.metadata.recordSet` gives the list of record set objects (with their `@id`)
- Each record set has fields, accessible as `.field` (with `@id`)

In [ ]:
# List available record sets and fields by @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found! Check metadata format or schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}, name={rs.get('name', '(no name)')}")
        fields = rs.get('field', [])
        if fields:
            for fld in fields:
                print(f"  Field: @id={fld['@id']}, name={fld.get('name', '(no name)')}")
        print("")
        # Preview first record
        try:
            rec_iter = dataset.records(record_set=rs['@id'])
            first = next(rec_iter)
            print("Sample record:")
            pprint.pprint(first)
        except Exception as e:
            print(f"Could not load sample record: {e}")
        print("----")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# --- Extract every record set as a DataFrame ---
dataframes = {}
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nDataFrame for RecordSet @id={rs_id}: Columns: {df.columns.tolist()}")
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by criteria, normalizing numeric fields, grouping data. All entity references are via `@id`.

- We'll select one record set (e.g., containing hormone levels or clinical features), choose a numeric field (by its `@id`), filter, normalize, and group as an example.

### Example: If record set contains a field for 'age', filter those with age > 25, normalize 'age', and group by 'infertility' status.

In [ ]:
# Choose the first RecordSet for demonstration
if record_set_ids:
    example_rs_id = record_set_ids[0]
    df = dataframes[example_rs_id]
    # Show available columns
    print(f"Columns: {df.columns.tolist()}")
    # Attempt to automatically find numeric and group fields
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.to_numeric(df[col], errors='coerce').notnull().any()]
    print(f"Numeric candidates: {numeric_candidates}")
    # Use heuristics to select field
    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]
    group_field_candidates = [col for col in df.columns if ('infertility' in col.lower() or 'phenotype' in col.lower() or df[col].dtype == object)]
    group_field = group_field_candidates[0] if group_field_candidates else None
    # Filter, normalize, group
    threshold = df[numeric_field].dropna().astype(float).mean() if not df[numeric_field].dropna().empty else 10
    filtered_df = df[df[numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    if not filtered_df.empty:
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, col_norm]].head())
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
else:
    print("No record sets or DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. All fields referenced via their `@id` (i.e., use DataFrame columns names, which are the Croissant field IDs).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot of numeric field in selected record set
if record_set_ids:
    df = dataframes[example_rs_id]
    plt.figure(figsize=(8,4))
    if numeric_field in df.columns:
        data = pd.to_numeric(df[numeric_field], errors='coerce')
        sns.histplot(data.dropna(), bins=6, kde=True)
        plt.title(f"Distribution of {numeric_field} in RecordSet {example_rs_id}")
        plt.xlabel(numeric_field)
        plt.show()

        plt.figure(figsize=(6,4))
        sns.boxplot(x=data.dropna())
        plt.title(f"Boxplot of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
    else:
        print(f"Numeric field {numeric_field} not found.")
else:
    print("No record sets available for visualization.")

## 6. Conclusion
In this notebook, we accessed the FAIR^2 Croissant dataset, listed its record sets and fields by `@id`, loaded tabular data for each set, and performed simple EDA and visualization using references compliant with schema standards. This process provides a reproducible path for downstream analysis and ensures clarity in data provenance and entity references.

**Key Takeaways:**
- The dataset is small and focused on clinical and genetic tabulations from 3 patients.
- Each entity (record set, field, column) was referenced via `@id`, ensuring clear tracing of Croissant entities.
- Limitations include scale and scope, but the structure enables academic exploration and reproducibility.

For more detailed analysis, consult the full Croissant metadata at the provided URL.